# Chapter 2.6 to 2.8: Token and Positional Embeddings

This notebook teaches the embedding ideas used in early transformer implementations, using small toy examples in PyTorch.

We will focus on five practical ideas:

1. Token embeddings with `nn.Embedding`
2. Positional embeddings
3. Combining token and positional embeddings
4. Batched embedding inputs
5. Broadcasting when adding positional embeddings to batches

The goal is not to train a model yet. The goal is to understand how token IDs become vectors that a transformer can process.

## 1. Imports

We only need PyTorch for this notebook.

We also set a random seed with `torch.manual_seed(...)`. This makes the random embedding values reproducible, so you should see the same numbers every time you run the notebook.

In [3]:
import torch
from torch import nn

# Set the random seed so randomly initialized embedding weights are reproducible.
torch.manual_seed(123)

# Print the PyTorch version so the environment is clear.
print("PyTorch version:", torch.__version__)

ModuleNotFoundError: No module named 'torch'

## Important Vocabulary

Before writing embedding code, we need four key terms.

**`vocab_size`** means the number of unique tokens the model knows about. If token IDs go from `0` to `9`, then `vocab_size = 10`.

**`embedding_dim`** means the length of the dense vector assigned to each token. If `embedding_dim = 4`, each token ID becomes a vector with 4 numbers.

**`context_length`** means the maximum number of token positions the model handles at once. If `context_length = 6`, the model has position embeddings for positions `0, 1, 2, 3, 4, 5`.

**`batch_size`** means how many sequences are processed together. A batch size of `2` means we feed two token sequences into the model at the same time.

## Intuition: Token ID to Dense Vector

Text cannot go directly into a neural network. Neural networks operate on numbers.

A tokenizer converts text into integer token IDs. For example, a fake sequence might look like this:

```text
"I like cats" -> [2, 5, 8]
```

But token IDs like `2`, `5`, and `8` are just labels. The number `8` is not automatically four times more meaningful than `2`.

An embedding layer solves this by turning each token ID into a learnable dense vector:

```text
token ID 2 -> [0.12, -0.30, 0.44, 0.91]
token ID 5 -> [-0.72, 0.08, 1.15, -0.21]
token ID 8 -> [0.33, 0.47, -0.62, 0.10]
```

During training, the model adjusts these vectors so useful relationships can be represented in the embedding space.

## 2. Single Sequence Token Embeddings

First, we create a small toy vocabulary and a token embedding layer.

Important shape idea:

```text
input token IDs shape:        [num_tokens]
output token embeddings shape:[num_tokens, embedding_dim]
```

Why does the shape change?

Each integer token ID is replaced by one dense vector of length `embedding_dim`.

In [ ]:
# Number of unique fake tokens in our toy vocabulary.
vocab_size = 10

# Each token ID will be mapped to a vector with this many numbers.
embedding_dim = 4

# Create a token embedding table with shape [vocab_size, embedding_dim].
token_embedding_layer = nn.Embedding(vocab_size, embedding_dim)

# A single fake input sequence containing 6 token IDs.
input_ids = torch.tensor([2, 4, 1, 7, 5, 3])

print("vocab_size:", vocab_size)
print("embedding_dim:", embedding_dim)
print("input_ids:")
print(input_ids)
print("input_ids shape:", input_ids.shape)
print("\nEmbedding table shape:", token_embedding_layer.weight.shape)
print("Why [vocab_size, embedding_dim]? One row per token ID, one vector per row.")

Now we pass the token IDs through the embedding layer.

The input contains 6 token IDs. Each token ID becomes a vector of length 4.

So the output shape becomes:

```text
[6] -> [6, 4]
```

In [ ]:
# Look up the embedding vector for each token ID in the input sequence.
token_embeddings = token_embedding_layer(input_ids)

print("input_ids shape:", input_ids.shape)
print("token_embeddings shape:", token_embeddings.shape)
print("\nWhy did the shape change?")
print("Each of the 6 token IDs was replaced with a vector of length 4.")
print("\nToken embeddings:")
print(token_embeddings)

An embedding layer is basically a lookup table.

The token ID selects a row from the embedding matrix.

For example, token ID `2` selects row `2` from `token_embedding_layer.weight`.

In [ ]:
# Show the full embedding table.
print("Full token embedding table:")
print(token_embedding_layer.weight)

# Compare the first embedded token with row 2 of the embedding table.
print("\nFirst input token ID:", input_ids[0].item())
print("Embedding returned for first input token:")
print(token_embeddings[0])
print("\nRow 2 in the embedding table:")
print(token_embedding_layer.weight[2])
print("\nThey match because token ID 2 looks up row 2.")

## 3. Positional Embeddings

Transformers process all tokens in a sequence largely in parallel. Unlike recurrent neural networks, a basic transformer block does not automatically know which token came first, second, third, and so on.

That is a problem because word order matters.

For example:

```text
"dog bites man"
"man bites dog"
```

These sentences contain the same words, but the meaning changes because the positions changed.

A positional embedding gives each position its own learnable vector. This tells the model where each token appears in the sequence.

We now create a positional embedding layer.

Important shape idea:

```text
position IDs shape:          [context_length]
position embeddings shape:   [context_length, embedding_dim]
```

The positional embedding vectors must have the same `embedding_dim` as the token embeddings because we will add them together later.

In [ ]:
# The maximum number of token positions our toy model can handle.
context_length = 6

# Create one learnable position vector for each position in the context.
position_embedding_layer = nn.Embedding(context_length, embedding_dim)

# Create position IDs: 0, 1, 2, 3, 4, 5.
position_ids = torch.arange(context_length)

# Look up the position embedding vector for each position ID.
position_embeddings = position_embedding_layer(position_ids)

print("context_length:", context_length)
print("embedding_dim:", embedding_dim)
print("position_ids:")
print(position_ids)
print("position_ids shape:", position_ids.shape)
print("\nPosition embedding table shape:", position_embedding_layer.weight.shape)
print("position_embeddings shape:", position_embeddings.shape)
print("\nWhy [6, 4]? There are 6 positions, and each position gets a vector of length 4.")
print("\nPosition embeddings:")
print(position_embeddings)

## 4. Combining Token and Positional Embeddings

Token embeddings answer:

```text
What token is this?
```

Positional embeddings answer:

```text
Where is this token in the sequence?
```

We combine them by adding the two tensors element by element.

The shapes must match:

```text
token_embeddings shape:      [6, 4]
position_embeddings shape:   [6, 4]
combined shape:              [6, 4]
```

The shape does not change during addition because we are combining two tensors with the same shape.

In [ ]:
# Add token information and position information together.
combined_embeddings = token_embeddings + position_embeddings

print("token_embeddings shape:", token_embeddings.shape)
print("position_embeddings shape:", position_embeddings.shape)
print("combined_embeddings shape:", combined_embeddings.shape)
print("\nWhy did the shape stay [6, 4]?")
print("We added two [6, 4] tensors element by element, so the result is also [6, 4].")
print("\nCombined embeddings:")
print(combined_embeddings)

Let's inspect one position manually.

For position `0`, the combined vector is:

```text
token embedding at position 0 + position embedding for position 0
```

This gives one vector that contains both token identity and position information.

In [ ]:
# Choose one sequence position to inspect.
pos = 0

print("Position being inspected:", pos)
print("Token ID at this position:", input_ids[pos].item())
print("\nToken embedding at this position:")
print(token_embeddings[pos])
print("\nPosition embedding for this position:")
print(position_embeddings[pos])
print("\nCombined embedding at this position:")
print(combined_embeddings[pos])
print("\nManual check:")
print(token_embeddings[pos] + position_embeddings[pos])

## 5. Batch Processing

Real models usually process many sequences at once.

A batch of token IDs has shape:

```text
[batch_size, context_length]
```

For example, if `batch_size = 2` and `context_length = 6`, then the input shape is `[2, 6]`.

After token embedding lookup, each token ID becomes a vector of length `embedding_dim`:

```text
[batch_size, context_length] -> [batch_size, context_length, embedding_dim]
[2, 6] -> [2, 6, 4]
```

In [ ]:
# Number of sequences processed together.
batch_size = 2

# A batch containing 2 fake sequences, each with 6 token IDs.
batch_input_ids = torch.tensor([
    [2, 4, 1, 7, 5, 3],
    [6, 8, 0, 2, 9, 1]
])

print("batch_size:", batch_size)
print("context_length:", context_length)
print("embedding_dim:", embedding_dim)
print("\nbatch_input_ids:")
print(batch_input_ids)
print("batch_input_ids shape:", batch_input_ids.shape)
print("\nWhy [2, 6]? There are 2 sequences, and each sequence contains 6 token IDs.")

Now we pass the full batch through the same token embedding layer.

PyTorch applies the lookup to every token ID in the batch.

The result has one vector per token per sequence.

In [ ]:
# Look up token embeddings for every token ID in every sequence.
batch_token_embeddings = token_embedding_layer(batch_input_ids)

print("batch_input_ids shape:", batch_input_ids.shape)
print("batch_token_embeddings shape:", batch_token_embeddings.shape)
print("\nWhy did the shape change from [2, 6] to [2, 6, 4]?")
print("There are 2 sequences, 6 tokens per sequence, and each token becomes a vector of length 4.")
print("\nBatch token embeddings:")
print(batch_token_embeddings)

## 6. Broadcasting Explanation

For a batch, token embeddings have shape:

```text
[batch_size, context_length, embedding_dim]
[2, 6, 4]
```

Position embeddings have shape:

```text
[context_length, embedding_dim]
[6, 4]
```

When we add them, PyTorch uses broadcasting.

Visually, PyTorch treats the position embeddings as if they had an extra leading batch dimension:

```text
batch_token_embeddings: [2, 6, 4]
position_embeddings:       [6, 4]
                         -> [1, 6, 4]
broadcasted over batch:  -> [2, 6, 4]
```

The same position vectors are added to every sequence in the batch.

This makes sense because position `0` means the first token position for every sequence, position `1` means the second token position for every sequence, and so on.

Let's add batched token embeddings and position embeddings.

The final shape remains `[2, 6, 4]` because each token in each sequence now has a vector containing both token and position information.

In [ ]:
# Add position information to every sequence in the batch.
batch_combined_embeddings = batch_token_embeddings + position_embeddings

print("batch_token_embeddings shape:", batch_token_embeddings.shape)
print("position_embeddings shape:", position_embeddings.shape)
print("batch_combined_embeddings shape:", batch_combined_embeddings.shape)
print("\nWhy does this work?")
print("PyTorch broadcasts [6, 4] to [1, 6, 4], then reuses it across the batch dimension.")
print("\nBatch combined embeddings:")
print(batch_combined_embeddings)

We can make the broadcasting more explicit by manually adding a batch dimension with `unsqueeze(0)`.

This changes the position embedding shape from `[6, 4]` to `[1, 6, 4]`.

That leading `1` means: one set of position embeddings, reusable across all sequences in the batch.

In [ ]:
# Add an explicit batch dimension to the position embeddings.
position_embeddings_with_batch_dim = position_embeddings.unsqueeze(0)

# Add using the explicitly expanded-looking shape.
batch_combined_embeddings_explicit = batch_token_embeddings + position_embeddings_with_batch_dim

print("position_embeddings shape:", position_embeddings.shape)
print("position_embeddings_with_batch_dim shape:", position_embeddings_with_batch_dim.shape)
print("batch_token_embeddings shape:", batch_token_embeddings.shape)
print("batch_combined_embeddings_explicit shape:", batch_combined_embeddings_explicit.shape)
print("\nAre implicit and explicit broadcasting results equal?")
print(torch.allclose(batch_combined_embeddings, batch_combined_embeddings_explicit))

Here is a visual way to think about the addition.

For a batch with 2 sequences and 6 positions:

```text
Sequence 0 token vectors:  positions 0 1 2 3 4 5
Sequence 1 token vectors:  positions 0 1 2 3 4 5

Position vectors:          positions 0 1 2 3 4 5
                           added to both sequences
```

In tensor shape form:

```text
[2, 6, 4]
+  [6, 4]
----------
[2, 6, 4]
```

The missing batch dimension in `[6, 4]` is automatically treated as size `1`, then copied logically across the two batch rows. PyTorch does not need to physically copy the data to perform this operation.

Let's inspect one position in both sequences.

Notice that the same position embedding is added at position `3` in both sequences, even though the token IDs at that position may be different.

In [ ]:
# Pick a position to inspect across both sequences.
inspect_position = 3

print("Position inspected:", inspect_position)
print("Position embedding used for both sequences:")
print(position_embeddings[inspect_position])

for sequence_index in range(batch_size):
    print("\nSequence index:", sequence_index)
    print("Token ID at this position:", batch_input_ids[sequence_index, inspect_position].item())
    print("Token embedding:")
    print(batch_token_embeddings[sequence_index, inspect_position])
    print("Combined embedding:")
    print(batch_combined_embeddings[sequence_index, inspect_position])

## 7. Final Combined Example

Now we put everything together in a compact, realistic flow.

The transformer input embedding step usually follows this pattern:

1. Start with token IDs of shape `[batch_size, context_length]`.
2. Convert token IDs to token embeddings of shape `[batch_size, context_length, embedding_dim]`.
3. Create position IDs of shape `[context_length]`.
4. Convert position IDs to position embeddings of shape `[context_length, embedding_dim]`.
5. Add token embeddings and position embeddings.
6. Get final input embeddings of shape `[batch_size, context_length, embedding_dim]`.

In [ ]:
# Reset the seed to make this final example reproducible on its own.
torch.manual_seed(123)

# Define toy model dimensions.
vocab_size = 12
embedding_dim = 5
context_length = 4
batch_size = 3

# Fake batch of token IDs: 3 sequences, each with 4 tokens.
token_ids = torch.tensor([
    [1, 3, 5, 7],
    [2, 4, 6, 8],
    [0, 9, 10, 11]
])

# Create token and position embedding layers.
token_emb = nn.Embedding(vocab_size, embedding_dim)
pos_emb = nn.Embedding(context_length, embedding_dim)

# Convert token IDs into dense token vectors.
tok_embeds = token_emb(token_ids)

# Create position IDs for this context length.
pos_ids = torch.arange(context_length)

# Convert position IDs into dense position vectors.
pos_embeds = pos_emb(pos_ids)

# Add token and position embeddings using broadcasting.
input_embeddings = tok_embeds + pos_embeds

print("vocab_size:", vocab_size)
print("embedding_dim:", embedding_dim)
print("context_length:", context_length)
print("batch_size:", batch_size)
print("\ntoken_ids:")
print(token_ids)
print("token_ids shape:", token_ids.shape)
print("\ntok_embeds shape:", tok_embeds.shape)
print("Why [3, 4, 5]? 3 sequences, 4 tokens each, 5 numbers per token vector.")
print("\npos_ids:")
print(pos_ids)
print("pos_ids shape:", pos_ids.shape)
print("pos_embeds shape:", pos_embeds.shape)
print("Why [4, 5]? 4 positions, 5 numbers per position vector.")
print("\ninput_embeddings shape:", input_embeddings.shape)
print("Why [3, 4, 5]? Position embeddings are added to every sequence in the batch.")
print("\nFinal input embeddings:")
print(input_embeddings)

## 8. Exercises

Try these small exercises to check your understanding.

### Exercise 1

Change `embedding_dim` from `5` to `3` in the final example.

Before running the cell, predict the shapes of:

```text
tok_embeds
pos_embeds
input_embeddings
```

### Exercise 2

Change `batch_size` by adding one more sequence to `token_ids`.

Question: Which dimension of `input_embeddings` changes?

### Exercise 3

Change `context_length` from `4` to `6`, then update every sequence in `token_ids` so each one has 6 token IDs.

Question: Which dimensions of `tok_embeds`, `pos_embeds`, and `input_embeddings` change?

### Exercise 4

Inspect `token_emb.weight[3]` and compare it with every place where token ID `3` appears in `tok_embeds`.

Question: Why do they match?

### Exercise 5

Try this code:

```python
wrong_pos_embeds = torch.randn(4, 6)
tok_embeds + wrong_pos_embeds
```

Question: Why does this fail when `tok_embeds` has last dimension `5`?

Hint: addition requires compatible dimensions. The embedding dimensions must match.

## 9. Key Takeaways

- Token IDs are integer labels, not meaningful numeric values by themselves.
- `nn.Embedding(vocab_size, embedding_dim)` creates a learnable lookup table.
- A token embedding turns each token ID into a dense vector.
- `vocab_size` is the number of possible token IDs.
- `embedding_dim` is the number of values in each embedding vector.
- `context_length` is the number of token positions the model can process.
- `batch_size` is the number of sequences processed together.
- Positional embeddings are needed because transformers need explicit information about token order.
- Token embeddings and positional embeddings are added because they have the same `embedding_dim`.
- For batched inputs, token embeddings usually have shape `[batch_size, context_length, embedding_dim]`.
- Position embeddings usually have shape `[context_length, embedding_dim]`.
- Broadcasting allows `[context_length, embedding_dim]` to be added across the batch dimension.
- The final combined embeddings are what we pass into the transformer blocks.

## Final Summary

The embedding step converts discrete token IDs into continuous vectors.

A token embedding tells the model what token is present. A positional embedding tells the model where that token appears. Adding them creates a single vector per token position that contains both kinds of information.

For one sequence:

```text
[context_length] -> [context_length, embedding_dim]
```

For a batch:

```text
[batch_size, context_length] -> [batch_size, context_length, embedding_dim]
```

This is the representation shape that transformer blocks expect.